In [ ]:
import os

# Même méthode que 02_nettoyage_historique : on remonte à la racine du dépôt,
# donc les chemins sont les mêmes que dans src/ (ET-04 : pas de chemin absolu)
while not os.path.isdir("src"):
    os.chdir("..")

import pandas as pd

historique = pd.read_parquet("data/processed/historique_gazole_idf.parquet")
stations = pd.read_parquet("data/processed/stations_idf.parquet")
print("historique :", historique.shape, "| stations :", stations.shape)

# Même type des deux côtés, sinon "75001001" ne retrouve jamais 75001001
print("type de la clé :", historique["id_station"].dtype, "/", stations["id_station"].dtype)

# validate="many_to_one" lèvera une erreur si une station apparaît deux fois dans le flux
print("id en double dans le flux :", stations["id_station"].duplicated().sum())

# Ampleur attendue d'ANO-13, calculée avant la jointure pour la vérifier après
ids_historique = pd.Series(historique["id_station"].unique())
absents_du_flux = ~ids_historique.isin(stations["id_station"])
print("stations dans l'historique :", len(ids_historique))
print("absentes du flux :", absents_du_flux.sum(), f"({absents_du_flux.mean():.1%})")

In [ ]:
# ANO-12 : avant D-18, on compare le département déduit du code postal (historique)
# au département officiel du flux, uniquement pour les stations présentes dans les deux
departements_flux = stations[["id_station", "departement"]].rename(
    columns={"departement": "departement_flux"}
)
comparaison = historique.merge(
    departements_flux, on="id_station", how="inner", validate="many_to_one"
)
differents = comparaison[comparaison["departement"] != comparaison["departement_flux"]]

print("stations concernées :", differents["id_station"].nunique())
print("relevés concernés :", len(differents), f"({len(differents) / len(historique):.2%})")
print(differents[["id_station", "code_postal", "departement", "departement_flux"]].drop_duplicates())

In [ ]:
from src.cleaning import joindre_historique_stations

historique_joint = joindre_historique_stations(historique, stations)
print("relevés :", len(historique), "->", len(historique_joint))

# Contrôle EF-06 : on doit retrouver les 121 stations du bloc 1
sans_ville = historique_joint[historique_joint["ville"].isna()]
print("stations non retrouvées :", sans_ville["id_station"].nunique())
print("relevés concernés :", len(sans_ville), f"({len(sans_ville) / len(historique_joint):.1%})")

In [ ]:
# Contrat D-11 après D-18 : une seule colonne departement, jamais vide, uniquement l'IDF
assert "departement_flux" not in historique_joint.columns
assert historique_joint["departement"].notna().all(), "département vide après D-18"
assert historique_joint["departement"].isin(DEPARTEMENTS_IDF).all(), "département hors IDF"
print(historique_joint["departement"].value_counts().sort_index())

# Contrôle D-18 : 94160001 a un code postal du 94 mais est dans le 75 selon le flux
print(historique_joint.loc[historique_joint["id_station"] == "94160001", "departement"].unique())

historique_joint.to_parquet("data/processed/historique_joint_idf.parquet", index=False)

# On relit le fichier pour vérifier que rien n'a été perdu à l'enregistrement
relu = pd.read_parquet("data/processed/historique_joint_idf.parquet")
assert relu.shape == historique_joint.shape
print("enregistré :", relu.shape)